In [10]:
import utils
import numpy as np
import pandas as pd
import os
from scipy.stats import chisquare

import matplotlib.pyplot as plt
import seaborn as sns # Loads in the theme
sns.set_theme()

pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_rows', None)

In [11]:
# Make output folders
output_dir = os.path.join('..', '..','pre-processed', 'data_for_modelling')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [12]:
# Preparation
experiment_output_path = os.path.join('..', '..', '..', 'phd_1_task', 'experiment_output', 'behavioural_files')
#experiment_output_path = os.path.join('..', '..', '..', 'phd_1_task', 'experiment_output_verena')

In [13]:
unique_subject_ids = list(range(801, 850))
unique_sessions = [1]

recent_results = utils.analysis_functions.list_files_with_date_and_subject_id(experiment_output_path, unique_subject_ids, unique_sessions)
print(recent_results)

sub-816 did not complete session-01
sub-817 did not complete session-01
sub-818 did not complete session-01
sub-819 did not complete session-01
sub-820 did not complete session-01
sub-821 did not complete session-01
sub-822 did not complete session-01
sub-823 did not complete session-01
sub-824 did not complete session-01
sub-825 did not complete session-01
sub-826 did not complete session-01
sub-827 did not complete session-01
sub-828 did not complete session-01
sub-829 did not complete session-01
sub-830 did not complete session-01
sub-831 did not complete session-01
sub-832 did not complete session-01
sub-833 did not complete session-01
sub-834 did not complete session-01
sub-835 did not complete session-01
sub-836 did not complete session-01
sub-837 did not complete session-01
sub-838 did not complete session-01
sub-839 did not complete session-01
sub-840 did not complete session-01
sub-841 did not complete session-01
sub-842 did not complete session-01
sub-843 did not complete ses

In [14]:
# Opens all the files and combines them into one dataframe
combined_results_dataframe = utils.analysis_functions.combine_result_files(recent_results)
if 'verena' in experiment_output_path:
    combined_dataframe = combined_results_dataframe[~combined_results_dataframe['stimuli_type'].isin([2, 4])]
    
print(combined_results_dataframe)

       trial  emotional_cue_time response objectively_correct  \
0          0        1.725887e+09      NaN                Late   
1          1        1.725887e+09     down                Even   
2          2        1.725887e+09     down                Even   
3          3        1.725887e+09       up                Even   
4          4        1.725888e+09       up                Even   
5          5        1.725888e+09     down                Even   
6          6        1.725888e+09     down                Even   
7          7        1.725888e+09       up                Even   
8          8        1.725888e+09     down                True   
9          9        1.725888e+09     down                True   
10        10        1.725888e+09       up                True   
11        11        1.725888e+09     down               False   
12        12        1.725888e+09       up                True   
13        13        1.725888e+09     down                True   
14        14        1.725

# Check congruency, valence and volatility

In [15]:
# Check a block's congruency
combined_results_dataframe['congruency'] = combined_results_dataframe.apply(utils.analysis_functions.check_congruency, args=(True,), axis=1)

# Check a block's valence
combined_results_dataframe['valence'] = combined_results_dataframe.apply(utils.analysis_functions.check_valence, args=(True,), axis=1)

# Check a block's volatility
combined_results_dataframe = combined_results_dataframe[combined_results_dataframe['probability_condition'] != 50]
combined_results_dataframe = combined_results_dataframe.groupby('stimuli_type', group_keys=False).apply(lambda x: utils.analysis_functions.check_volatility(x, True))
combined_results_dataframe = combined_results_dataframe.sort_values(by=['stimuli_type','trial'], ascending=[True, True])
print(combined_results_dataframe[['volatility','probability_condition']])
combined_results_dataframe = combined_results_dataframe.sort_values(by=['subject_id', 'session', 'stimuli_type', 'trial'], ascending=[True, True, True, True])
combined_results_dataframe.reset_index(drop=True, inplace=True)
print(combined_results_dataframe[['subject_id', 'session', 'congruency', 'valence', 'volatility']])

print(combined_results_dataframe['valence'].unique())
print(combined_results_dataframe['congruency'].unique())
print(combined_results_dataframe['volatility'].unique())

     volatility  probability_condition
0             0                     20
1             1                     20
2             1                     80
3             1                     80
4             1                     80
5             0                     80
6             1                     80
7             0                     20
8             0                     20
9             1                     80
10            0                     80
11            0                     80
12            1                     20
13            0                     20
14            1                     80
15            1                     80
16            1                     80
17            0                     80
18            1                     80
19            1                     80
20            1                     80
21            0                     80
22            0                     80
23            0                     80
24            0          

/var/folders/bh/ld02p1_550d75y0x32dmpb540000gp/T/ipykernel_14483/3460074756.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combined_results_dataframe = combined_results_dataframe.groupby('stimuli_type', group_keys=False).apply(lambda x: utils.analysis_functions.check_volatility(x, True))


# Recode existing variables

In [16]:
combined_results_dataframe = combined_results_dataframe.dropna(subset=['response'])

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'up': 2, 'down': 1},
                                                                                    'response',
                                                                                    'response_int')

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {3: 1, 1: 2},
                                                                                    'stimuli_type',
                                                                                    'stimuli_type')

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'True': 1, 'False': 0},
                                                                                    'subjectively_correct',
                                                                                    'subjectively_correct_int')

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'True': 1, 'False': 0},
                                                                                    'objectively_correct',
                                                                                    'objectively_correct_int')

combined_results_dataframe['subject_id'] = combined_results_dataframe['subject_id'].str.replace(r'\D', '', regex=True)

combined_results_dataframe['session'] = combined_results_dataframe['session'].str.replace(r'\D', '', regex=True)

# Now export the data

In [18]:
combined_results_dataframe.to_csv(f'{output_dir}/combined_dataframe_binary_pilot_hard.csv', sep=';', index_label='index')